# Dev — Eval unification smoke

Sanity-check the new unified eval entry point `python -m prism.eval.scalability_evaluation` on a single E5 test graph.

**Checkpoint:** `outputs/e5_pe_sweep/e5_llm_r5pbg901` (plain LLM, 4-bit, e5 PE sweep — text-only baseline).

**Graph:** `data/gen/e5_graph_oriented_data/split/test_graphs/data_gen_009.json` (22 nodes, 10 tasks).

**Command used to produce `results/dev/data_gen_009.json`:**

```bash
python -m prism.eval.scalability_evaluation \
    --checkpoint outputs/e5_pe_sweep/e5_llm_r5pbg901 \
    --graphs /home/shared/GREP-PRISM/data/gen/e5_graph_oriented_data/split/test_graphs/data_gen_009.json \
    --output results/dev --four-bit --device -1
```

In [1]:
import json
import pandas as pd
from pathlib import Path

RESULT_PATH = Path('../results/dev/data_gen_009.json')
result = json.loads(RESULT_PATH.read_text())

{
    'checkpoint': result['checkpoint'].split('/')[-1],
    'graph_file': result['graph_file'].split('/')[-1],
    'architecture': result['architecture'],
    'text_edge_list': result['text_edge_list'],
    'accuracy': result['accuracy'],
    'num_correct': result['num_correct'],
    'num_samples': result['num_samples'],
}

{'checkpoint': 'e5_llm_r5pbg901',
 'graph_file': 'data_gen_009.json',
 'architecture': 'llm',
 'text_edge_list': 'present',
 'accuracy': 0.7,
 'num_correct': 7,
 'num_samples': 10}

## Per-sample breakdown

Each task in the graph file becomes one row. `graph_name` is the new field added during the eval-unification refactor — it's stamped on every sample so flattening multiple graphs into one DataFrame stays traceable. `idx` is now the true per-sample index (the old `eval_checkpoint_on_graphs.py` had an `idx=0` bug).

In [2]:
rows = []
for s in result['samples']:
    rows.append({
        'idx': s['idx'],
        'graph_name': s['graph_name'],
        'task': s['task'][:90],
        'answer_key': s['answer_key'],
        'formatted': s['formatted'],
        'plan_keyword': s['plan_keyword'],
        'correct': s['correct'],
        'terminated_by': s['terminated_by'],
        'error': s['error'],
    })

df = pd.DataFrame(rows)
df

,idx,graph_name,task,answer_key,formatted,plan_keyword,correct,terminated_by,error
0,0,data_gen_009,Starting from the main passenger concourse whe...,(?i)\byes\b,True,False,False,answer,None
1,1,data_gen_009,Is there a connected route from the main passe...,(?i)\bconcourse_1\b.*\bdeicing_pad_1\b,True,True,True,answer,None
2,2,data_gen_009,"From elsewhere in the airfield, determine whic...",(?i)\bcargo_office_1\b,True,False,False,answer,None
3,3,data_gen_009,Can the robot travel from the maintenance hang...,(?i)\bhangar_1\b.*\bfuel_farm_1\b.*\bfuel_tank...,True,True,True,answer,None
4,4,data_gen_009,Is there a route from the boarding gate area w...,(?i)\bgate_1\b.*\bcargo_ramp_1\b,True,True,True,answer,None
5,5,data_gen_009,Find out whether this airport complex includes...,(?i)\byes\b,True,True,True,answer,None
6,6,data_gen_009,Is there a route from the main passenger conco...,(?i)\bconcourse_1\b.*\bhangar_1\b.*\bmaintenan...,True,False,False,answer,None
7,7,data_gen_009,From the freight warehouse used to store incom...,(?i)\bwarehouse_1\b.*\bapron_1\b,True,True,True,answer,None
8,8,data_gen_009,Is there a connected path from the deicing pad...,(?i)\bdeicing_pad_1\b.*\bgate_1\b,True,True,True,answer,None
9,9,data_gen_009,Starting from the cargo office area that manag...,(?i)\bcargo_office_1\b.*\bfuel_tank_1\b,True,True,True,answer,None


In [3]:
# Quick summary by correctness flag
df[['formatted', 'plan_keyword', 'correct']].agg(['sum', 'mean'])

,formatted,plan_keyword,correct
sum,10.0,7.0,7.0
mean,1.0,0.7,0.7


## Failure cases

Tasks where the model produced a well-formatted answer but the answer-key regex didn't match the plan field.

In [4]:
for s in result['samples']:
    if s['correct']:
        continue
    print(f"=== task {s['idx']} (graph_name={s['graph_name']}) ===")
    print(f"  task:        {s['task']}")
    print(f"  answer_key:  {s['answer_key']}")
    print(f"  formatted:   {s['formatted']}")
    print(f"  plan_kw:     {s['plan_keyword']}")
    plan = (s.get('response') or {}).get('plan', '<no plan>')
    print(f"  plan:        {plan}")
    print()

=== task 0 (graph_name=data_gen_009) ===
  task:        Starting from the main passenger concourse where travelers arrive from check-in, can the robot move in a single step directly out to the apron area where the operational fuel truck is parked, without passing through any other space first?
  answer_key:  (?i)\byes\b
  formatted:   True
  plan_kw:     False
  plan:        [['answer', 'The robot cannot move directly from the main passenger concourse to the apron area. It must pass through taxiway_1 first.']]

=== task 2 (graph_name=data_gen_009) ===
  task:        From elsewhere in the airfield, determine which specific cargo-side area contains the forklift used for loading freight near the administrative offices of the cargo operation.
  answer_key:  (?i)\bcargo_office_1\b
  formatted:   True
  plan_kw:     False
  plan:        [['answer', 'The forklift used for loading freight near the administrative offices of the cargo operation is located in the cargo_ramp_1 area.']]

=== task 6

## Notes

- This is a single-graph cross_eval run; no permutation seeds. The output JSON lives at `results/dev/data_gen_009.json` and follows the cross_eval shape consumed by `eval_viewer.html` and the judge-eval skill.
- With `--permutation-seed` supplied the same script switches to the transferability layout (`<output>/perm_<seed>/<ckpt>_<graph>.json` + per-seed `<ckpt>_summary.json`).
- `use_icl` defaults to `True` at the argparse layer (5 SPINE worked examples in the planner prompt). `--use-icl false` falls back to a 1-example variant.